<div align="center">

# Week 08: Visual Design and Emotion: Grouping, Layout, First and Last Impressions

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-08) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/Week08_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `hex_rgb`, which turns a colour code into three numbers, and stores the lattice ratios of the floor dots and the wait patterns of the kiosk. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


def hex_rgb(code):
    """'#2F5D8A' -> [47, 93, 138]"""
    return [int(code[i:i + 2], 16) for i in (1, 3, 5)]

# The floor dots of lab Part A: the ratios |b|/|a| of the rectangular lattices
RATIOS = [1.0, 1.1, 1.2, 1.3, 1.4]
# The waits of lab Part D in seconds after the five taps of a booking
PATTERNS = {"late": [0.5, 0.5, 0.5, 0.5, 4.0], "early": [4.0, 0.5, 0.5, 0.5, 0.5],
            "middle": [0.5, 0.5, 4.0, 0.5, 0.5], "flat": [1.2] * 5}
print("Setup complete. Running in Colab:", IN_COLAB, "| lattice ratios:", RATIOS,
      "| wait patterns:", list(PATTERNS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 8.1": (lambda: 4000, 1e-9),
    "Exercise 8.2": (lambda: 0.05, 1e-9),
    "Exercise 8.3": (lambda: int((screen == hex_rgb("#2F5D8A")).all(axis=2).sum()), 1e-9),
    "Exercise A": (lambda: 1 / (1 + math.exp(-6.6 * 0.3) + 2 * math.exp(-6.6 * (math.sqrt(1 + 1.3 ** 2) - 1))), 1e-6),
    "Exercise B": (lambda: (6 * 2 - 4 * 4) / (4 * 4), 1e-6),
    "Exercise C": (lambda: 1.3 * math.hypot(23, 34), 1e-6),
    "Exercise D": (lambda: (8 + 4) / 2, 1e-6),
    "Exercise E": (lambda: float(STAGE8["peak-end of the waits, new (s)"]), 1e-6),
}

## Python step 8: Images as arrays

A screen is a grid of coloured points, and NumPy stores such a grid as an array of numbers. This step builds small images from rectangles, reads their colours and computes statistics of their pixels, the basis of the layout measures of this week &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-1), [2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "PATTERNS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### An image is an array of numbers

In [ ]:
# Python practice: a small example for learning the language
img = np.zeros((4, 6, 3), dtype=np.uint8)   # 4 rows, 6 columns, 3 colours
img[:, :] = (255, 255, 255)                     # every pixel white
img[1:3, 2:5] = (47, 93, 138)                   # a blue block
print("shape:", img.shape)
print("pixel at row 1, column 2:", img[1, 2], "| at row 0, column 0:", img[0, 0])
plt.figure(figsize=(3, 2))
plt.imshow(img)
plt.show()

An image is an array with three axes: Rows from top to bottom, columns from left to right and the three colour channels red, green and blue. The type `uint8` holds whole numbers from 0 to 255. `img[1:3, 2:5]` selects rows 1 and 2 and columns 2 to 4, and assigning three numbers colours all of them.

### Colour channels and masks

In [ ]:
# Python practice: a small example for learning the language
red = img[..., 0]                      # the red channel of every pixel
print(red)
is_blue = (img == (47, 93, 138)).all(axis=2)
print("blue pixels:", is_blue.sum(), "of", is_blue.size)

`img[..., 0]` keeps all rows and columns and takes channel 0, the red values. Comparing the image with one colour compares each channel. The method `.all(axis=2)` asks whether all three channels match, so the result is a mask of `True` and `False` with one value per pixel.

### Drawing rectangles into an image

In [ ]:
# Python practice: a small example for learning the language
def draw(rects, width, height, background="#FFFFFF"):
    """An image of the given size with rectangles (x, y, w, h, colour)."""
    image = np.zeros((height, width, 3), dtype=np.uint8)
    image[:, :] = hex_rgb(background)
    for x, y, w, h, colour in rects:
        image[y:y + h, x:x + w] = hex_rgb(colour)
    return image

screen = draw([(0, 0, 120, 16, "#2F5D8A"), (10, 28, 70, 8, "#1E2433"),
               (10, 56, 100, 18, "#2F5D8A")], 120, 80)
plt.figure(figsize=(3, 2))
plt.imshow(screen)
plt.show()

A rectangle at `x, y` with width `w` and height `h` covers the rows `y` to `y + h - 1` and the columns `x` to `x + w - 1`. It is therefore the slice `[y:y + h, x:x + w]`. The function `hex_rgb` of the setup turns a colour code such as `"#2F5D8A"` into the three numbers 47, 93 and 138.

### Statistics of an image

In [ ]:
# Python practice: a small example for learning the language
grey = screen @ np.array([0.299, 0.587, 0.114])   # brightness of each pixel
print("mean brightness:", round(float(grey.mean()), 1))
print("share of dark pixels:", round(float((grey < 100).mean()), 3))
step = np.abs(np.diff(grey, axis=1))               # change to the right neighbour
print("pixels where the brightness jumps:", int((step > 20).sum()))

The operator `@` multiplies each pixel's three channels with three weights and adds them, which gives its brightness. The function `np.diff(grey, axis=1)` subtracts each pixel from its right neighbour, so large values mark the edges of the rectangles.

**Quick check.** Why does a colour image need three numbers per pixel, while a brightness image needs one?

<details><summary>Answer</summary>

A screen mixes each colour from red, green and blue light, so a colour needs three intensities. Brightness is a single weighted sum of the three.

</details>

### Exercises for Python step 8

**Exercise 8.1.** Create a white image of 50 rows and 80 columns with `np.zeros` and store its number of pixels, rows times columns, in `n_pixels`.

In [ ]:
n_pixels = None   # your answer

check("Exercise 8.1", n_pixels)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
white = np.zeros((50, 80, 3), dtype=np.uint8) + 255
n_pixels = white.shape[0] * white.shape[1]

check("Exercise 8.1", n_pixels)

**Exercise 8.2.** Draw a rectangle of 20 by 10 pixels into a white image of 80 by 50 pixels with `draw`. Store the share of the image that it covers in `share`.

In [ ]:
share = None   # your answer

check("Exercise 8.2", share)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
im = draw([(5, 5, 20, 10, "#1E2433")], 80, 50)
share = float((im == hex_rgb("#1E2433")).all(axis=2).mean())

check("Exercise 8.2", share)

**Exercise 8.3.** Count the pixels of `screen` whose colour is `"#2F5D8A"` and store the number in `n_blue`.

In [ ]:
n_blue = None   # your answer

check("Exercise 8.3", n_blue)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_blue = int((screen == hex_rgb("#2F5D8A")).all(axis=2).sum())

check("Exercise 8.3", n_blue)

## 1. Grouping by proximity: Your floor dots &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-4)&#93;

*Lecture: [Gestalt: How the eye groups](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#gestalt)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "draw" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The pure distance law turns the Gestalt principle of proximity into numbers. Python predicts how often each direction of a dot lattice is seen and fits the attraction constant to your own choices.

**From the formulas to the code.** The pull of a direction $f(v) = e^{-\alpha(|v|/|a| - 1)}$ is `attraction(ratio, alpha)`. The probability of seeing the rows along $a$, $p(a) = 1 / \sum_v f(v)$, is the first value of `lattice_probs(r)`.

In [ ]:
ALPHA = 6.6     # a representative attraction constant of Kubovy and colleagues

def attraction(ratio, alpha=ALPHA):
    """f(v) = exp(-alpha (|v|/|a| - 1)), the pull of a direction relative to a."""
    return math.exp(-alpha * (ratio - 1))

def lattice_probs(r, alpha=ALPHA):
    """Probabilities of seeing the rows along a, b, c and d (rectangular lattice)."""
    diagonal = math.sqrt(1 + r ** 2)
    f = [1.0, attraction(r, alpha), attraction(diagonal, alpha),
         attraction(diagonal, alpha)]
    return [x / sum(f) for x in f]

for r in RATIOS + [1.5]:
    p = lattice_probs(r)
    print(f"|b|/|a| = {r:.1f}: p(a) = {p[0]:.3f}, p(b) = {p[1]:.3f}, "
          f"p(c) = p(d) = {p[2]:.3f}")

**Python note.** In a rectangular lattice the two diagonals $c$ and $d$ have the same length $\sqrt{|a|^2 + |b|^2}$, so they pull equally. Dividing each pull by the sum of all four turns the pulls into probabilities that add up to 1.

**Exercise A.** Store the probability $p(a)$ of a rectangular lattice with $|b|/|a| = 1.3$ and $\alpha = 6.6$ in `p_a`.

In [ ]:
p_a = None   # your answer

check("Exercise A", p_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_a = lattice_probs(1.3)[0]

check("Exercise A", p_a)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_LATTICE`. Left empty, the variable makes the code use a simulated class of 20 students with attraction constants around 6.6.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_LATTICE = """
"""

def simulated_choices(n_students=20, seed=RANDOM_STATE):
    """Each student sees every ratio six times and reports one direction."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        alpha = rng.normal(ALPHA, 1.0)
        for r in RATIOS:
            for _ in range(6):
                choice = rng.choice(list("abcd"), p=lattice_probs(r, alpha))
                rows.append([student, r, choice])
    return pd.DataFrame(rows, columns=["student", "ratio", "choice"])

if MY_LATTICE.strip():
    trials = pd.read_csv(io.StringIO(MY_LATTICE.strip()))
    source = "your own choices from the interactive lab"
else:
    trials = simulated_choices()
    source = "a simulated class of 20 students"
counts = trials.pivot_table(index="ratio", columns="choice", aggfunc="size",
                            fill_value=0).reindex(columns=list("abcd"), fill_value=0)
print("Data:", source)
print(counts)
x = counts.index.to_numpy() - 1
y = np.log((counts["b"] + 0.5) / (counts["a"] + 0.5)).to_numpy()
alpha_fit = -np.sum(x * y) / np.sum(x * x)
print(f"fitted attraction constant: {alpha_fit:.2f} (Kubovy and colleagues: 6.6)")

**Python note.** `pivot_table` counts the choices for each ratio, one column per direction. Adding 0.5 to each count keeps the logarithm finite when a direction was never chosen. The pure distance law is a straight line through the origin, $\ln(p(b)/p(a)) = -\alpha(|b|/|a| - 1)$, so its least-squares slope is $-\sum xy / \sum x^2$.

In [ ]:
# Figure: your choices and the pure distance law
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(counts.index, y, "o", color="#5A4BB5", label="observed")
rr = np.linspace(1.0, 1.45, 50)
ax[0].plot(rr, -alpha_fit * (rr - 1), color="#C98510",
           label=f"fitted, alpha = {alpha_fit:.1f}")
ax[0].plot(rr, -ALPHA * (rr - 1), ":", color="#585E6E", label="alpha = 6.6")
ax[0].set_xlabel("|b| / |a|")
ax[0].set_ylabel("ln p(b) / p(a)")
ax[0].legend()
share = counts.div(counts.sum(axis=1), axis=0)
for k, colour in zip("abcd", ["#5A4BB5", "#C98510", "#2E8B57", "#B53F2E"]):
    ax[1].plot(share.index, share[k], "o-", color=colour, label=f"chose {k}")
ax[1].set_xlabel("|b| / |a|")
ax[1].set_ylabel("share of the trials")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** `counts.div(counts.sum(axis=1), axis=0)` divides each row by its total, so the right panel shows the share of each direction for each ratio.

**Observe and reflect.** From which ratio on did you almost always see the rows of the shorter distance? What gap between two groups of buttons would you choose, compared with the gap inside a group?

## 2. Layout: Balance, edges and heat maps &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-5)&#93;

*Lecture: [Layout: Balance, alignment and where the eye lands](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#layout)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "alpha_fit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Balance and alignment can be computed from the rectangles of a screen, and a heat map from the points where people looked. Python computes the balance measure of Ngo and colleagues, counts the edges that elements share and sums Gaussian blobs into a heat map.

**From the formulas to the code.** The weight of a side $w_j = \sum_i a_{ij} d_{ij}$ is computed inside `balance`, which returns the balance measure $BM = 1 - (|BM_{vertical}| + |BM_{horizontal}|)/2$. The heat map $H(x, y) = \sum_k e^{-((x - x_k)^2 + (y - y_k)^2)/(2\sigma^2)}$ is `heat_map(fixations, W, H, sigma)`.

In [ ]:
def balance(rects, W, H):
    """Balance measure BM of Ngo and colleagues for rectangles (x, y, w, h, ...)."""
    wl = wr = wt = wb = 0.0
    for x, y, w, h, *_ in rects:
        a, cx, cy = w * h, x + w / 2, y + h / 2
        if cx < W / 2:
            wl += a * (W / 2 - cx)
        else:
            wr += a * (cx - W / 2)
        if cy < H / 2:
            wt += a * (H / 2 - cy)
        else:
            wb += a * (cy - H / 2)
    bm_v = (wl - wr) / max(wl, wr) if max(wl, wr) > 0 else 0.0
    bm_h = (wt - wb) / max(wt, wb) if max(wt, wb) > 0 else 0.0
    return 1 - (abs(bm_v) + abs(bm_h)) / 2, bm_v, bm_h

def edges(rects):
    """Distinct left edges and distinct top edges of the rectangles."""
    return len({r[0] for r in rects}), len({r[1] for r in rects})

EXAMPLE = [(0.5, 1.5, 4, 2), (5.5, 5.5, 3, 4)]     # x, y, w, h on a 10 x 10 frame
FIXED = [(0, 0, 360, 56, "#2F5D8A"), (16, 88, 220, 18, "#1E2433"),
         (16, 128, 328, 160, "#EEF2F7")]   # a small screen for later cells
bm, bm_v, bm_h = balance(EXAMPLE, 10, 10)
print(f"worked example: vertical {bm_v:.3f}, horizontal {bm_h:.3f}, BM {bm:.3f}")

**Python note.** `for x, y, w, h, *_ in rects` takes the first four values of each rectangle and collects the rest, such as a colour, in `_`. An element belongs to the side of its centre, as in the lecture.

**Exercise B.** An element of area 6 lies 2 units left of the centre line, and an element of area 4 lies 4 units right of it. Store $BM_{vertical}$ in `bm_v`.

In [ ]:
bm_v = None   # your answer

check("Exercise B", bm_v)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
bm_v = (6 * 2 - 4 * 4) / max(6 * 2, 4 * 4)

check("Exercise B", bm_v)

**Explore.** Move the sliders: The panel moves the blue button of a small screen and prints the balance measure. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(x=16, y=520):
    rects = FIXED + [(x, y, 160, 56, "#2F5D8A")]
    image = draw(rects, 360, 640)
    plt.figure(figsize=(2.6, 4.2))
    plt.imshow(image)
    plt.axvline(180, ls=":", color="#C98510")
    plt.axhline(320, ls=":", color="#C98510")
    plt.axis("off")
    plt.show()
    print(f"BM = {balance(rects, 360, 640)[0]:.3f}")

widgets.interact(explore, x=widgets.IntSlider(value=16, min=0, max=200, step=8),
                 y=widgets.IntSlider(value=520, min=300, max=584, step=8));

**Python note.** The library ipywidgets calls `explore` again each time a slider moves. The dotted lines are the central lines of the frame, from which the distances $d_{ij}$ are measured.

In [ ]:
def heat_map(fixations, W, H, sigma=39):
    """Sum of a Gaussian blob around each fixation, scaled to a maximum of 1."""
    yy, xx = np.mgrid[0:H, 0:W]
    heat = np.zeros((H, W))
    for fx, fy in fixations:
        heat += np.exp(-((xx - fx) ** 2 + (yy - fy) ** 2) / (2 * sigma ** 2))
    return heat / heat.max()

two = heat_map([(100, 200), (160, 200)], 360, 400)
raw = lambda x: (math.exp(-(x - 100) ** 2 / (2 * 39 ** 2))
                 + math.exp(-(x - 160) ** 2 / (2 * 39 ** 2)))
print("worked example: midpoint", round(raw(130), 3), "| at a fixation",
      round(raw(100), 3))
print("hottest column of the map:", int(two[200].argmax()))

**Python note.** `np.mgrid[0:H, 0:W]` returns two arrays with the row and the column of every pixel, so the Gaussian is computed for all pixels at once. The worked example uses the raw sum before scaling.

In [ ]:
# Figure: a heat map of simulated fixations on a screen
rng = np.random.default_rng(RANDOM_STATE)
targets = [(180, 28, 120, 14), (126, 97, 90, 8), (180, 208, 140, 70),
           (96, 548, 70, 22)]
weights = np.array([0.25, 0.2, 0.3, 0.25])
fix = []
for k in rng.choice(4, size=40, p=weights):
    cx, cy, sx, sy = targets[k]
    fix.append((cx + rng.uniform(-sx, sx), cy + rng.uniform(-sy, sy)))
fix = np.array(fix)
example = draw(FIXED + [(16, 520, 160, 56, "#2F5D8A")], 360, 640)
fig, ax = plt.subplots(1, 2, figsize=(5.4, 4.4))
ax[0].imshow(example)
ax[0].plot(fix[:, 0], fix[:, 1], ".", color="#C98510")
ax[1].imshow(example)
ax[1].imshow(heat_map(fix, 360, 640), cmap="hot", alpha=0.55)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

**Python note.** The fixations here are simulated around four elements. `imshow` with `alpha=0.55` lays the heat map over the screen, so the hot regions show where the eyes rested.

**Observe and reflect.** Where would you place the main button so that the balance measure rises? Does the most balanced position also make sense for a thumb on a phone?

## 3. First impressions: Your fifty milliseconds &#91;[7](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-7)&#93;

*Lecture: [First impressions in fifty milliseconds](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#first-impressions)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "heat_map" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** First impressions depend on properties that can be computed from the pixels of a design. Python rebuilds the twelve designs of the interactive lab as arrays, computes their colourfulness and their edge density, and compares them with your ratings.

**From the formulas to the code.** The opponent channels $rg = R - G$ and $yb = (R + G)/2 - B$ are `rg` and `yb`, and the colourfulness $M = \sigma_{rgyb} + 0.3\,\mu_{rgyb}$ is `colourfulness(image)`. The edge density is `edge_density(image)`.

In [ ]:
PALETTES = {
    "calm": ("#FFFFFF", "#2F5D8A", ["#2F5D8A"], "#E8EEF5"),
    "muted": ("#ECEDEF", "#5B6170", ["#9AA1AC", "#B7BCC4", "#7D8592"], "#D7DADF"),
    "colourful": ("#FFFFFF", "#E2533B", ["#E2533B", "#F2B134", "#3FA34D",
                  "#2F7DD1", "#8C4FC7", "#E0629B"], "#FFF4D6"),
    "loud": ("#14121F", "#FF2E88", ["#FF2E88", "#00E5FF", "#C6FF00", "#FFEA00",
                                    "#FF6D00"], "#3A2F5C")}
# palette, number of tiles, tiles on a grid, number of extra elements
SPECS = [("calm", 4, True, 0), ("calm", 6, True, 1), ("muted", 6, True, 0),
         ("colourful", 6, True, 0), ("colourful", 9, True, 2), ("loud", 6, True, 2),
         ("calm", 9, False, 3), ("muted", 12, False, 4), ("colourful", 12, False, 6),
         ("loud", 16, False, 8), ("calm", 16, True, 4), ("colourful", 4, False, 2)]

class LCG:
    """A tiny random generator with whole numbers, identical on every computer."""
    def __init__(self, seed):
        self.s = seed
    def random(self):
        self.s = self.s * 48271 % 2147483647
        return self.s / 2147483647
    def integer(self, lo, hi):
        return lo + int(self.random() * (hi - lo + 1))

def design(k, W=640, H=400):
    """The rectangles of design k (0 to 11) of the fifty-millisecond test."""
    name, n, aligned, extras = SPECS[k]
    bg, head, tiles, light = PALETTES[name]
    r = LCG(1000 + 37 * k)
    R = [(0, 0, W, H, bg), (0, 0, W, 52, head), (16, 14, 24, 24, light),
         (52, 20, 150, 12, light)]
    cols = 4 if n >= 12 else 3 if n >= 6 else 2
    rows = -(-n // cols)
    tw = (W - 32 - (cols - 1) * 16) // cols
    th = min(110, (H - 84 - (rows - 1) * 14) // rows)
    text = "#2B2F38" if name == "muted" else "#FFFFFF"
    for i in range(n):
        x, y = 16 + (i % cols) * (tw + 16), 68 + (i // cols) * (th + 14)
        w, h = tw, th
        if not aligned:
            x, y = x + r.integer(-22, 22), y + r.integer(-12, 12)
            w = int(w * (0.7 + 0.35 * r.random()))
            h = int(h * (0.75 + 0.3 * r.random()))
        R.append((x, y, w, h, tiles[i % len(tiles)]))
        for j in range(1 + i % 2):                      # one or two lines of text
            line_w = int((w - 20) * (0.75 - 0.25 * j))
            R.append((x + 10, y + h - 22 - 14 * j, line_w, 7, text))
    for e in range(extras):
        if e % 3 == 0:                                  # a banner
            x, y = r.integer(0, W - 220), r.integer(60, H - 70)
            w, h = r.integer(140, 220), r.integer(34, 60)
            R += [(x, y, w, h, tiles[(e + 2) % len(tiles)]),
                  (x + 8, y + 8, w - 40, 8, light)]
        elif e % 3 == 1:                                # a row of icons
            x, y = r.integer(0, W - 200), r.integer(60, H - 40)
            R += [(x + 38 * j, y, 26, 26, tiles[(e + j) % len(tiles)])
                  for j in range(5)]
        else:                                           # a ticker with text
            y = r.integer(60, H - 30)
            R.append((0, y, W, 20, light))
            R += [(10 + 104 * j, y + 6, 70 + r.integer(0, 20), 7,
                   "#2B2F38" if name != "loud" else "#FFFFFF") for j in range(6)]
    return [(max(0, x), max(0, y), min(w, W - max(0, x)), min(h, H - max(0, y)), c)
            for x, y, w, h, c in R]

def colourfulness(image):
    """Colourfulness M of Hasler and Suesstrunk for an RGB image (0 to 255)."""
    R, G, B = (image[..., i].astype(float) for i in range(3))
    rg, yb = R - G, 0.5 * (R + G) - B
    return float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))

def edge_density(image, threshold=20):
    """Share of pixels whose brightness jumps by more than the threshold."""
    grey = image @ np.array([0.299, 0.587, 0.114])
    jump = (np.abs(np.diff(grey, axis=1))[:-1, :]
            + np.abs(np.diff(grey, axis=0))[:, :-1])
    return float((jump > threshold).mean())

images = [draw(design(k)[1:], 640, 400, design(k)[0][4]) for k in range(12)]
measures = pd.DataFrame({"design": range(1, 13),
                         "palette": [s[0] for s in SPECS],
                         "elements": [len(design(k)) - 1 for k in range(12)],
                         "M": [round(colourfulness(im), 1) for im in images],
                         "edges": [round(edge_density(im), 4) for im in images]})
print(measures.to_string(index=False))

**Python note.** The class `LCG` makes the same sequence of pseudo-random numbers in Python and in the browser, so the designs here are exactly those of the interactive lab. `np.hypot(a, b)` computes $\sqrt{a^2 + b^2}$, and `-(-n // cols)` divides and rounds up.

**Exercise C.** Build the image of the worked example, two white and two blue pixels (`"#2F5D8A"`), as an array of 2 by 2 pixels, and store its colourfulness in `m_c`.

In [ ]:
m_c = None   # your answer

check("Exercise C", m_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
tiny = draw([(0, 1, 2, 1, "#2F5D8A")], 2, 2)
m_c = colourfulness(tiny)

check("Exercise C", m_c)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_FLASH`. Left empty, the variable makes the code use a simulated class of 20 students.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_FLASH = """
"""

def simulated_ratings(n_students=20, seed=RANDOM_STATE):
    """Ratings from 1 to 9: simpler designs and moderate colour rate higher."""
    rng = np.random.default_rng(seed)
    off_colour = (measures["M"] - 50).abs()
    base = (7.6 - 60 * measures["edges"] - 0.012 * off_colour).to_numpy()
    rows = []
    for student in range(1, n_students + 1):
        for rnd, noise in [(1, 1.2), (2, 0.8)]:
            for k in range(12):
                rating = int(np.clip(round(base[k] + rng.normal(0, noise)), 1, 9))
                rows.append([student, k + 1, rnd, rating])
    return pd.DataFrame(rows, columns=["student", "design", "round", "rating"])

if MY_FLASH.strip():
    flash = pd.read_csv(io.StringIO(MY_FLASH.strip()))
    source = "your own ratings from the interactive lab"
else:
    flash = simulated_ratings()
    source = "a simulated class of 20 students"
means = flash.pivot_table(index="design", columns="round", values="rating")
means.columns = ["50 ms", "long view"]
means = means.join(measures.set_index("design")[["M", "edges"]])
print("Data:", source)
if "shown_ms" in flash and flash["shown_ms"].notna().any():
    print("median time a flash stayed on the screen:",
          int(flash["shown_ms"].median()), "ms")
print(means.round(2))
print("r between 50 ms and the long view:",
      round(means["50 ms"].corr(means["long view"]), 2))
print("r of the long view with the edge density:",
      round(means["long view"].corr(means["edges"]), 2),
      "| with the colourfulness:", round(means["long view"].corr(means["M"]), 2))

**Python note.** The interactive lab records how long each flash really stayed on the screen, which depends on the refresh rate of the display. `pivot_table` averages the ratings of each design in each round, and `.corr` computes the Pearson correlation $r$ between two columns, from $-1$ to $1$.

In [ ]:
# Figure: the twelve designs and your ratings
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(means["50 ms"], means["long view"], color="#5A4BB5")
for d, row in means.iterrows():
    ax[0].annotate(str(d), (row["50 ms"], row["long view"]), fontsize=8,
                   xytext=(3, 3), textcoords="offset points")
ax[0].plot([1, 9], [1, 9], ":", color="#585E6E")
ax[0].set_xlabel("rating after 50 ms")
ax[0].set_ylabel("rating after a long view")
sc = ax[1].scatter(means["edges"], means["long view"], c=means["M"], cmap="viridis")
fig.colorbar(sc, ax=ax[1], label="colourfulness M")
ax[1].set_xlabel("edge density")
ax[1].set_ylabel("rating after a long view")
plt.tight_layout()
plt.show()

**Python note.** The dotted diagonal marks equal ratings in both rounds. In the right panel the colour of each point shows the colourfulness of the design.

**Observe and reflect.** Did your ratings after 50 ms agree with those after a long view? Which design surprised you, and which measure would have predicted your rating?

## 4. The remembered experience: Your waits at the kiosk &#91;[12](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#ref-12)&#93;

*Lecture: [Emotion: Beauty, usability and the remembered experience](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#emotion)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "measures" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** People remember an episode by its worst moment and its end more than by its total. Python computes the mean, the sum and the peak-end value of discomfort traces and compares them with your ratings of the bookings in the interactive lab.

**From the formulas to the code.** The peak-end value $PE = (\max_t x_t + x_{end})/2$ is `peak_end(trace)`, and the mean $\bar{x}$ is `np.mean(trace)`.

In [ ]:
def peak_end(trace):
    """The average of the worst moment and the last moment."""
    return (max(trace) + trace[-1]) / 2

for trace in [[2, 5, 8], [2, 5, 8, 4]]:
    print(trace, "mean", round(float(np.mean(trace)), 2), "| sum", sum(trace),
          "| peak-end", peak_end(trace))
waits = pd.DataFrame({"total (s)": {k: sum(v) for k, v in PATTERNS.items()},
                      "mean (s)": {k: round(float(np.mean(v)), 2)
                                   for k, v in PATTERNS.items()},
                      "peak-end (s)": {k: peak_end(v) for k, v in PATTERNS.items()}})
print(waits)

**Python note.** A dictionary comprehension such as `{k: sum(v) for k, v in PATTERNS.items()}` builds one value per pattern, and `pd.DataFrame` turns several such dictionaries into the columns of a table.

**Exercise D.** Store the peak-end value of the discomfort trace 2, 5, 8, 4 in `pe_d`.

In [ ]:
pe_d = None   # your answer

check("Exercise D", pe_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pe_d = peak_end([2, 5, 8, 4])

check("Exercise D", pe_d)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_WAITS`. Left empty, the variable makes the code use a simulated class of 20 students.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_WAITS = """
"""

def simulated_bookings(n_students=20, seed=RANDOM_STATE):
    """Ratings from 1 (annoying) to 9 (pleasant) that follow the peak-end value."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        for pattern, trace in PATTERNS.items():
            for _ in range(2):
                rating = 7.0 - 0.9 * peak_end(trace) + rng.normal(0, 0.9)
                rows.append([student, pattern, int(np.clip(round(rating), 1, 9))])
    return pd.DataFrame(rows, columns=["student", "pattern", "rating"])

if MY_WAITS.strip():
    bookings = pd.read_csv(io.StringIO(MY_WAITS.strip()))
    source = "your own ratings from the interactive lab"
else:
    bookings = simulated_bookings()
    source = "a simulated class of 20 students"
table = bookings.groupby("pattern")["rating"].mean().to_frame("mean rating")
table = table.join(waits).sort_values("peak-end (s)")
print("Data:", source)
print(table.round(2))
print("r of the ratings with the peak-end value:",
      round(table["mean rating"].corr(table["peak-end (s)"]), 2))

**Python note.** All four patterns wait 6 s in total, so the total and the mean cannot explain different ratings. `join` adds the columns of the table `waits` by the name of the pattern.

In [ ]:
# Figure: the waits of the four patterns and your ratings
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for (name, trace), colour in zip(PATTERNS.items(),
                                 ["#B53F2E", "#2E8B57", "#C98510", "#5A4BB5"]):
    ax[0].plot(range(1, 6), trace, "o-", color=colour,
               label=f"{name}: peak-end {peak_end(trace):.2f} s")
ax[0].set_xlabel("tap of the booking")
ax[0].set_ylabel("wait after the tap (s)")
ax[0].legend(fontsize=8)
ax[1].scatter(table["peak-end (s)"], table["mean rating"], color="#5A4BB5")
for name, row in table.iterrows():
    ax[1].annotate(name, (row["peak-end (s)"], row["mean rating"]), fontsize=9,
                   xytext=(4, 4), textcoords="offset points")
ax[1].set_xlabel("peak-end value of the waits (s)")
ax[1].set_ylabel("mean rating, 1 to 9")
plt.tight_layout()
plt.show()

**Python note.** The left panel draws the four traces of waits, the right one the mean rating of each pattern against its peak-end value.

**Observe and reflect.** Which pattern did you rate lowest, and was it the one with the long wait at the end? Would you rather wait 4 s at the start or at the end of a booking, and why?

## 5. The clinic app, stage 8

*Lecture: [The clinic app, stage 8: Redesigned screens with a clear visual hierarchy](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#the-clinic-app-stage-8)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "peak_end" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 8 compares the summary screen of the current app with the redesign. Python computes the balance, the distinct edges, the colourfulness and the edge density of both screens. It also simulates fixations with a simple salience model and compares the waits of the two booking flows.

In [ ]:
CURRENT = [  # the summary screen of the current app: (x, y, w, h, colour)
    (0, 0, 360, 24, "#30343C"), (0, 24, 360, 56, "#C0392B"),
    (12, 36, 32, 32, "#FFFFFF"), (56, 46, 150, 12, "#FFFFFF"),
    (8, 92, 344, 64, "#F4D03F"),
    (20, 106, 220, 10, "#7A5C00"), (20, 126, 160, 10, "#7A5C00"),
    (14, 174, 200, 11, "#30343C"), (22, 196, 176, 11, "#30343C"),
    (10, 218, 224, 11, "#30343C"), (18, 240, 120, 11, "#30343C"),
    (26, 262, 92, 11, "#30343C"), (12, 298, 100, 44, "#2E86C1"),
    (30, 315, 64, 10, "#FFFFFF"), (124, 304, 110, 40, "#E67E22"),
    (142, 319, 72, 10, "#FFFFFF"), (246, 296, 102, 48, "#8E44AD"),
    (270, 315, 54, 10, "#FFFFFF"), (20, 372, 228, 40, "#ECEFF1"),
    (30, 387, 150, 10, "#8E979F"), (262, 372, 84, 40, "#27AE60"),
    (280, 387, 48, 10, "#FFFFFF"), (0, 540, 360, 24, "#16A085"),
    (10, 548, 90, 8, "#FFFFFF"), (112, 548, 120, 8, "#FFFFFF"),
    (246, 548, 96, 8, "#FFFFFF")] + [
    (16 + 68 * i, 588, 36, 36, c) for i, c in
    enumerate(["#E74C3C", "#F39C12", "#2ECC71", "#3498DB", "#9B59B6"])]
NEW = [  # the redesign on a grid of 16 dp
    (0, 0, 360, 24, "#E8EEF5"), (0, 24, 360, 56, "#2F5D8A"),
    (16, 42, 20, 20, "#FFFFFF"), (110, 46, 140, 12, "#FFFFFF"),
    (324, 42, 20, 20, "#FFFFFF"), (80, 104, 200, 18, "#1E2433"),
    (16, 144, 328, 208, "#EEF2F7"),
    (32, 172, 120, 11, "#585E6E"), (192, 172, 120, 12, "#1E2433"),
    (32, 220, 104, 11, "#585E6E"), (192, 220, 104, 12, "#1E2433"),
    (32, 268, 88, 11, "#585E6E"), (192, 268, 128, 12, "#1E2433"),
    (32, 316, 96, 11, "#585E6E"), (192, 316, 96, 12, "#1E2433"),
    (16, 376, 328, 48, "#EEF2F7"), (80, 394, 200, 12, "#585E6E"),
    (0, 520, 360, 120, "#F4F6F9"), (16, 536, 328, 56, "#2F5D8A"),
    (148, 558, 64, 12, "#FFFFFF"), (148, 608, 64, 12, "#2F5D8A")]
BOOK = {"current": (262, 372, 84, 40), "new": (16, 536, 328, 56)}
WAITS = {"current": [0.4, 0.4, 0.5, 0.4, 3.0], "new": [3.0, 0.4, 0.4, 0.5, 0.4]}

def salience(rects, background="#FFFFFF"):
    """A simple salience model: size times contrast with the background."""
    luminance = np.array([0.299, 0.587, 0.114])
    bg = np.array(hex_rgb(background)) @ luminance
    out = []
    for x, y, w, h, c in rects:
        rgb = np.array(hex_rgb(c), float)
        saturation = (rgb.max() - rgb.min()) / 255
        contrast = abs(rgb @ luminance - bg) / 255 + 0.5 * saturation
        out.append(math.sqrt(w * h) * contrast)
    return np.array(out)

def simulated_fixations(rects, n=60, seed=7):
    """Fixations that land on elements in proportion to their salience."""
    rng = np.random.default_rng(seed)
    p = salience(rects)
    pts = []
    for k in rng.choice(len(rects), size=n, p=p / p.sum()):
        x, y, w, h, c = rects[k]
        pts.append((x + w * (0.1 + 0.8 * rng.random()) + rng.normal(0, 6),
                    y + h * (0.1 + 0.8 * rng.random()) + rng.normal(0, 6)))
    return np.array(pts)

STAGE8 = {}
for name, rects in [("current", CURRENT), ("new", NEW)]:
    image = draw(rects, 360, 640)
    fix = simulated_fixations(rects)
    bx, by, bw, bh = BOOK[name]
    on_book = ((fix[:, 0] >= bx) & (fix[:, 0] <= bx + bw)
               & (fix[:, 1] >= by) & (fix[:, 1] <= by + bh))
    STAGE8[f"balance BM, {name}"] = round(balance(rects, 360, 640)[0], 3)
    STAGE8[f"distinct left and top edges, {name}"] = edges(rects)
    STAGE8[f"colourfulness M, {name}"] = round(colourfulness(image), 1)
    STAGE8[f"edge density, {name}"] = round(edge_density(image), 4)
    on_share = round(float(on_book.mean()), 3)
    STAGE8[f"simulated fixations on the button, {name}"] = on_share
    STAGE8[f"mean wait per step, {name} (s)"] = round(float(np.mean(WAITS[name])), 2)
    STAGE8[f"peak-end of the waits, {name} (s)"] = peak_end(WAITS[name])
for key, value in STAGE8.items():
    print(f"{key:<48} {value}")

**Python note.** Both screens use the functions of Sections 2 to 4 without changes: `balance`, `edges`, `colourfulness`, `edge_density`, `heat_map` and `peak_end`. The salience model and its fixations are a simulation, not a measurement, and Week 9 records real fixations and ratings.

In [ ]:
# Figure: the two summary screens with simulated heat maps
fig, ax = plt.subplots(1, 4, figsize=(10, 4.6))
for i, (name, rects) in enumerate([("current", CURRENT), ("new", NEW)]):
    image = draw(rects, 360, 640)
    ax[2 * i].imshow(image)
    ax[2 * i].set_title(f"{name}: BM {STAGE8[f'balance BM, {name}']}", fontsize=9)
    ax[2 * i + 1].imshow(image)
    ax[2 * i + 1].imshow(heat_map(simulated_fixations(rects), 360, 640), cmap="hot",
                         alpha=0.55)
    share = STAGE8[f"simulated fixations on the button, {name}"]
    ax[2 * i + 1].set_title(f"on the button: {share:.0%}", fontsize=9)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

**Python note.** Each screen appears twice: As drawn, and with the heat map of its simulated fixations. `:.0%` writes a share as a whole percentage.

**Exercise E.** Store the peak-end value of the waits of the new booking flow, as kept in `STAGE8`, in `pe_e`.

In [ ]:
pe_e = None   # your answer

check("Exercise E", pe_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pe_e = STAGE8["peak-end of the waits, new (s)"]

check("Exercise E", pe_e)

**Observe and reflect.** Which measure changed most between the two screens? Which of them would you check first with patients in Week 9?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE8" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-08/"
               "NB08_visual_design_and_emotion.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below has a main screen, given as rectangles. It also has the waits after the five steps of a frequent task, as it is now and in a redesign. The functions of this week compute the balance, the edges, the colourfulness and the peak-end values after one change of a name.

The screens and the waits are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {
    "clinic": ("book an appointment", NEW, WAITS["current"], WAITS["new"]),
    "banking": ("send money",
                [(0, 0, 360, 72, "#0B3D91"), (16, 96, 328, 120, "#E8EEF5"),
                 (16, 240, 160, 48, "#0B3D91"), (184, 240, 160, 48, "#E8EEF5"),
                 (16, 560, 328, 56, "#0B3D91")],
                [0.3, 0.3, 0.4, 0.3, 4.5], [2.0, 0.3, 0.4, 0.3, 0.6]),
    "e_government": ("upload a document",
                     [(0, 0, 360, 64, "#B53F2E"), (24, 96, 300, 20, "#1E2433"),
                      (8, 140, 250, 180, "#F4D03F"), (270, 140, 80, 80, "#2ECC71"),
                      (40, 560, 140, 48, "#B53F2E")],
                     [0.5, 0.6, 0.5, 0.4, 6.0], [0.5, 6.0, 0.5, 0.4, 0.4]),
    "food_delivery": ("order a meal",
        [(0, 0, 360, 64, "#E2533B"), (16, 88, 160, 120, "#F2B134"),
         (184, 88, 160, 120, "#3FA34D"), (16, 224, 160, 120, "#2F7DD1"),
         (184, 224, 160, 120, "#E0629B"), (16, 560, 328, 56, "#E2533B")],
        [0.3, 0.4, 0.3, 0.5, 3.5], [0.3, 0.4, 3.5, 0.5, 0.3]),
    "course_registration": ("register for a course",
        [(0, 0, 360, 56, "#5A4BB5"), (16, 80, 220, 16, "#1E2433"),
         (16, 112, 328, 300, "#EEF2F7"), (200, 560, 144, 48, "#5A4BB5")],
        [0.4, 0.4, 0.5, 0.4, 5.0], [5.0, 0.4, 0.5, 0.4, 0.4])}

task, rects, now, redesign = APPS[APP]
image = draw(rects, 360, 640)
print(f"{APP}: {task}")
print(f"  main screen: BM {balance(rects, 360, 640)[0]:.3f}, edges {edges(rects)}, "
      f"M {colourfulness(image):.1f}")
print(f"  waits now {now}: mean {np.mean(now):.2f} s, "
      f"peak-end {peak_end(now):.2f} s")
print(f"  redesign {redesign}: mean {np.mean(redesign):.2f} s, "
      f"peak-end {peak_end(redesign):.2f} s")

**Python note.** Each entry holds the task, the rectangles of the main screen and the waits of the current flow and of the redesign. The redesigns move the long wait away from the end or shorten it.

**Observe and reflect.** Does the redesign of your app shorten the total wait, or does it only move the long wait? Which element of its main screen would you move to raise the balance?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Take a screenshot of a dialog of your development environment, describe its elements as rectangles and compute its balance and its distinct edges. |
| Web development | Compute the colourfulness and the edge density of the home pages of three websites from screenshots loaded as arrays, and rank them. |
| Mobile development | Move the main button of an app screen to three positions and compare the balance measure with how easily a thumb reaches each position. |
| Game development | Draw the heads-up display of a game as rectangles and simulate fixations with the salience model of Section 5. Which element draws the eye first? |
| Embedded systems and the Internet of Things | Measure the waits of a smart device after each step of its setup and reorder them so that the setup ends quickly. |
| Data science | Collect first-impression ratings of ten dashboards from five classmates and correlate them with the colourfulness and the edge density. |
| Cybersecurity | Design two versions of a security warning and compute their colourfulness. Discuss why an alarming design can also train people to ignore it. |
| Health informatics | Record the waits of a booking on a hospital website and compute their peak-end value. Propose an order of the steps with a better end. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-08/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-08).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>